In [4]:
import pandas as pd

df = pd.read_parquet(r'C:\Users\jhchoei\Downloads\07005127_20260123_SuddenDrop_45oC_05CP_RPT+100cyc+RPT_no1290_No2 (1).parquet')

In [5]:
df

,No,StepNo,State,StepType,DataSelect,Reserved1,GradeCode,Mode,IndexFrom,IndexTo,...,GasCO2,GasTemp,GasAH,GasBaseline,GasTVOC,GasEthanol,GasH2,TestMode,IsRptCapa,IsRptDcir
0,14,2,2,Rest,1,248,0,None,0,121,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RPT,False,False
1,14,5,2,Discharge,1,248,0,CC,122,510,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RPT,False,False
2,14,6,2,Rest,1,248,0,None,511,648,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RPT,False,False
3,14,7,2,Charge,1,248,0,CCCV,649,1946,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RPT,True,False
4,14,8,2,Rest,1,248,0,None,1947,2132,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RPT,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,14,41,2,Discharge,1,248,0,CC,261052,261152,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RPT,False,True
423,14,42,2,Rest,1,248,0,None,261153,262894,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RPT,False,False
424,14,43,2,Charge,1,248,0,CC,262895,262995,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RPT,False,False
425,14,44,2,Rest,1,248,0,None,262996,264738,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RPT,False,False


In [4]:
df['ChCode']

0        64
1        65
2        64
3        66
4        64
       ... 
1057     65
1058     64
1059     65
1060     64
1061    152
Name: ChCode, Length: 1062, dtype: int64

In [5]:
df['Code']

0          Time Complete
1       Voltage Complete
2          Time Complete
3       Current Complete
4          Time Complete
              ...       
1057    Voltage Complete
1058       Time Complete
1059    Voltage Complete
1060       Time Complete
1061       Time Complete
Name: Code, Length: 1062, dtype: object

In [6]:
import sys
# sys.path에 상위 폴더(..)를 추가합니다.
sys.path.append("..")

from src.voltaflow.connectors.db import DbConnector
from psycopg2 import Error
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import os

host = os.getenv("DB_HOST")
database_mart = "mart_db"
database_req = "req_sub_db"
database_account = "accounts_db"
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
port = os.getenv("DB_PORT")

def get_email_data():
    """
    Fetches data by joining subscription_data with account_data, and then with pipelinequeue_tb.
    """
    try:
        # Connect to req_sub_db for subscription_data
        req_db_conn = DbConnector(host, database_req, user, password, port)
        req_db_conn.connect()
        subscription_query = "SELECT cell_id, data_requester, subscription, exp_id FROM subscription_data;"
        subscription_data = req_db_conn.fetch_data(subscription_query, as_dataframe=True)
        subscription_data = subscription_data.rename(columns={"data_requester": "name"})
        # Connect to accounts_db for account_data
        account_db_conn = DbConnector(host, database_account, user, password, port)
        account_db_conn.connect()
        account_query = "SELECT name, email FROM account_data;"
        account_data = account_db_conn.fetch_data(account_query, as_dataframe=True)
        
        # Merge subscription_data and account_data
        merged_data = subscription_data.merge(account_data, on="name", how="left")
        
        # Connect to mart_db for pipelinequeue_tb
        mart_db_conn = DbConnector(host, database_mart, user, password, port)
        mart_db_conn.connect()
        pipeline_query = """
        SELECT exp_id, test_title, download_required_yn, last_download_datetime
        FROM pipeline_queue_tb;
        """
        pipeline_data = mart_db_conn.fetch_data(pipeline_query, as_dataframe=True)
        
        # Merge with pipelinequeue_tb data
        email_data = merged_data.merge(pipeline_data, on="exp_id", how="left")
        
    except (Exception, Error) as error:
        print(f"데이터베이스 작업 중 오류 발생: {error}")
        if req_db_conn:
            req_db_conn.rollback()
        if account_db_conn:
            account_db_conn.rollback()
        if mart_db_conn:
            mart_db_conn.rollback()
        
    finally:
        if req_db_conn:
            req_db_conn.disconnect()
        if account_db_conn:
            account_db_conn.disconnect()
        if mart_db_conn:
            mart_db_conn.disconnect()
    
    return email_data

In [161]:
df = get_email_data()

데이터베이스 'req_sub_db'에 성공적으로 연결되었습니다.
데이터베이스 'accounts_db'에 성공적으로 연결되었습니다.
데이터베이스 'mart_db'에 성공적으로 연결되었습니다.
커서가 닫혔습니다.
데이터베이스 연결이 종료되었습니다.
커서가 닫혔습니다.
데이터베이스 연결이 종료되었습니다.
커서가 닫혔습니다.
데이터베이스 연결이 종료되었습니다.


In [162]:
df

,cell_id,name,subscription,exp_id,email,test_title,download_required_yn,last_download_datetime
0,UDKTF10102,admin,Y,EXP_250507_00837,blmt2024@lgensol.com,20250501_JF2S_CCV_T55_CPC_1_1_12CP_SOC0~100_UD...,False,2025-11-05 17:49:26
1,DA05B19192,admin,Y,EXP_240704_01166,blmt2024@lgensol.com,20240507_JF1_MP_SII_JIS_45_0.5CCCV_0.5CC_DA05B...,False,NaT
2,EA23135723,admin,Y,EXP_250527_00393,blmt2024@lgensol.com,20250527_JF1_MP_45_0.75CP_SOC5~100_EA23135723,False,2025-11-05 18:58:21
3,CC27B18056,최정현,Y,EXP_250528_00069,jhchoei@lgensol.com,20250528_XXXXX_XX_T25_SC_JF1_PV1st_025CP_CC27B...,False,2025-11-06 16:31:06
4,CC27B18470,최정현,Y,EXP_250528_00071,jhchoei@lgensol.com,20250528_XXXXX_XX_T25_SC_JF1_PV1st_05CP_CC27B1...,False,2025-11-06 16:31:34
...,...,...,...,...,...,...,...,...
328,CC27B18470,조혁준,Y,EXP_240821_00046,hyukjun_jo@lgensol.com,20240821_JF1_DV2nd_no.2_CC27B18470_SOC0-100_0....,False,2025-11-05 18:00:03
329,CC27B18470,조혁준,Y,EXP_240531_00037,hyukjun_jo@lgensol.com,20240531_JF1_DV2nd_no.2_CC27B18470_SOC0-100_0....,False,2025-11-05 18:01:59
330,CC27B18470,조혁준,Y,EXP_231030_00027,hyukjun_jo@lgensol.com,20231030_JF1_PV_M_no.2_CC27B18470_SOC0-100_0.5...,False,2025-11-05 18:20:35
331,CC27B18470,조혁준,Y,EXP_230411_00056,hyukjun_jo@lgensol.com,20230411_JF1_PV_M_no.2_CC27B18470_SOC0-100_0.5...,False,2025-11-05 18:33:21


In [163]:
df = df.drop_duplicates(subset=['cell_id', 'exp_id','email'])

In [155]:
for email in df['email'].unique():
    temp = df[(df['email'] == email) & (df['subscription'] =='Y')]
    name = temp['name'].values[0]
    print(email, name)
    row_num_total = len(temp)
    row_num_required = len(temp[temp['download_required_yn'] == True])
    
    if row_num_required == 0 or name == 'admin':
        continue
            # List of experiments with download_required_yn = True
            
    if name != "조혁준":
        continue
    
    required_experiments = temp[temp['download_required_yn'] == True]['test_title'].tolist()
    experiments_list = "\n".join([f"-. {test_title}" for test_title in required_experiments])
    message = f"""{name} 님이 구독한 데이터 파이프라인 현황입니다.
구독중인 {row_num_total}개의 실험 중 이미 완료된 실험을 제외한 {row_num_required} 개 실험 업데이트 완료되었습니다.
#########업데이트 완료된 실험명#########
{experiments_list}
                """
    print(message)

blmt2024@lgensol.com admin
jhchoei@lgensol.com 최정현
seoheekang@lgensol.com 강서희
pyojh120@lgensol.com 표정호
sangwon.oh@lgensol.com 오상원
hyukjun_jo@lgensol.com 조혁준
조혁준 님이 구독한 데이터 파이프라인 현황입니다.
구독중인 5개의 실험 중 이미 완료된 실험을 제외한 2 개 실험 업데이트 완료되었습니다.
#########업데이트 완료된 실험명#########
-. 20250725_JF2S_CV_ORT_45_05CP_UDKRF20329_No1
-. 20250724_JF2S_CV_ORT_45_05CP_UDKRF20321_No8
                
taemin.goh@lgensol.com 고태민


In [99]:
temp

,cell_id,name,subscription,email,exp_id,download_required_yn,last_download_datetime
1115,UDKTF10115,고태민,Y,taemin.goh@lgensol.com,EXP_250507_00840,False,2025-11-05 21:28:35
1116,UDKTF10115,고태민,Y,taemin.goh@lgensol.com,EXP_250328_00758,False,2025-11-05 21:28:38
1117,UDKTF10115,고태민,Y,taemin.goh@lgensol.com,EXP_250312_00332,False,2025-11-05 22:33:46


In [ ]:
    """
    
    """

In [43]:
import requests
import json
import pandas as pd
import logging
import time
from datetime import  datetime
import os

header_kr = { 
    'Accept': 'application/json, text/javascript, */*; q=0.01',
    'Accept-Encoding': 'gzip, deflate',
    'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
    'Cache-Control': 'no-cache',
    'Connection': 'keep-alive',
    'Content-Length': '479',
    'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
    'Cookie': 'L-VISITOR=z2qurp1uebcoa6; JSESSIONID=542798FFA6087D23582E25EBA128DA80.6134f0ddc45e87161; InitechEamUURL=http%3A%2F%2Ftoscn.lgensol.com%3A8009%2Ftos%2Fmain.do; InitechEamRTOA=1; CLP=; InitechEamCookieEnc=T; LGCHEM_NC_FLAG=TRUE; ep_mode=F; InitechEamNCFlag=F; language=ko; change=Y; InitechEamUID=1mL53LU3fyaHIWvC6qPkMg%3D%3D; InitechEamUTOA=1; InitechEamUPID=XQDvTAgJ4afUQUwvgr1wJA%3D%3D; eWLanguage=ko-KR; tempLGDPID=hyukjun_jo; InitechEamDomain=LGENSOL; SCOUTER=z2bd0rldnhe3l8; leftMenuType=list; _ga=GA1.2.521480946.1681885360; _ga_JMHFCNW1FS=GS2.1.s1747959829$o32$g1$t1747959848$j41$l0$h0$ds4wgRbOLl7W1quhcvsYoxmV5slhbgmyuhg; _ga_QMD5NW261X=GS2.1.s1747959830$o31$g1$t1747959848$j0$l0$h0; InitechEamUIP=EaIGJz9JlvRTXewdIhl5Pg%3D%3D; InitechEamULAT=1748310792; InitechEamUHMAC=0SbTDvsBWhpZP7fJjhlMJhSpJ5iVj48xJpjQygsv2b0%3D; engpuid=pps1W+dAfqoY+W6Al2Qfytulp9ocriXZHmThkwuCDXs=; LENA-UID=17224de2.638bf38482291',
    'Host': 'toskr.lgensol.com:8009',
    'Origin': 'http://toskr.lgensol.com:8009',
    'Pragma': 'no-cache',
    'Referer': 'http://toskr.lgensol.com:8009/tos/testResultDownload/retrieveTstRltDldList.do?_selectedMenuId=353',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36',
    'X-Requested-With': 'XMLHttpRequest'
}

header_cn = { 
    'Accept': 'application/json, text/javascript, */*; q=0.01',
    'Accept-Encoding': 'gzip, deflate',
    'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
    'Cache-Control': 'no-cache',
    'Connection': 'keep-alive',
    'Content-Length': '496',
    'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
    'Cookie': 'L-VISITOR=z6drvdmv45p3ji; JSESSIONID=D52068096290FD9504B5C9632C4B60B6.5814bece043f87161; InitechEamUURL=http%3A%2F%2Ftoscn.lgensol.com%3A8009%2Ftos%2Fmain.do; InitechEamRTOA=1; CLP=; InitechEamCookieEnc=T; LGCHEM_NC_FLAG=TRUE; ep_mode=F; InitechEamNCFlag=F; language=ko; change=Y; InitechEamUID=1mL53LU3fyaHIWvC6qPkMg%3D%3D; InitechEamUTOA=1; InitechEamUPID=XQDvTAgJ4afUQUwvgr1wJA%3D%3D; eWLanguage=ko-KR; tempLGDPID=hyukjun_jo; InitechEamDomain=LGENSOL; installRetryCookie=2; _ga=GA1.2.521480946.1681885360; _ga_JMHFCNW1FS=GS2.1.s1747959829$o32$g1$t1747959848$j41$l0$h0$ds4wgRbOLl7W1quhcvsYoxmV5slhbgmyuhg; _ga_QMD5NW261X=GS2.1.s1747959830$o31$g1$t1747959848$j0$l0$h0; InitechEamUIP=EaIGJz9JlvRTXewdIhl5Pg%3D%3D; InitechEamULAT=1748310792; InitechEamUHMAC=0SbTDvsBWhpZP7fJjhlMJhSpJ5iVj48xJpjQygsv2b0%3D; engpuid=pps1W+dAfqoY+W6Al2Qfytulp9ocriXZHmThkwuCDXs=; LENA-UID=c28b0916.638bed54b0739; leftMenuType=list',
    'Host': 'toscn.lgensol.com:8009',
    'Origin': 'http://toscn.lgensol.com:8009',
    'Pragma': 'no-cache',
    'Referer': 'http://toscn.lgensol.com:8009/tos/testResultDownload/retrieveTstRltDldList.do?_selectedMenuId=310',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36',
    'X-Requested-With': 'XMLHttpRequest'
}

In [46]:
def request_test_results(site, sub ,payload):
    if sub == "retrieveTstRltFileList":
        base_url = "http://tos{}.lgensol.com:8009/tos/testResultDownload/{}.ajax"
    else: # 'retrieveTestResultList', 'requestTestResultData
        base_url = "http://tos{}.lgensol.com:8009/tos/test/resultCdms/{}.ajax"
    try:
        res = requests.post(base_url.format(site, sub),
                    headers=header_kr if site == 'kr' else header_cn,
                    data=payload,
                    timeout=5)
    except Exception as e:
        print(f"error by {e}")
        return None
    
    return res

In [47]:
cell_id = "20240531_JF1_DV2nd_no.2_BE27B13035_SOC0-100_0.25CP_25oC"
site='cn'
payload = {"searchTestTitle": cell_id,
            "searchUserName": "",}
res = request_test_results(site, "retrieveTestResultList", payload)

In [48]:
for row in res.json()['rows']:
    print(row)

{'customRowSize': [10, 20, 50], 'imSysMgrYn': 'N', 'iamJobTesterLeaderYn': 'N', 'autoCompleteSearchExceptStr': '', 'pageSize': 10, 'deptNm': 'ESS.개발.Cell.Cell개발2팀', 'iamJobTesterYn': 'N', 'rsltCode': 'FAIL', 'scheduleJobEqYn': 'N', 'autoCompleteSelectedStr': '', 'autoCompleteYn': 'N', 'languageCd': 'cn', 'groupingView': False, 'testFileLastUploadDatetime': '2025-10-29 14:38:45', 'rsltMssg': '', 'paging': True, 'iamTosTestLeaderYn': 'N', 'lastChnnEndDatetime': '2024-09-09 16:26:04.0', 'manufacturerCd': 'EQT18', 'testJobId': '20240531011793', 'mobileDevice': False, 'testStartDatetime': '2024-05-31 14:37:42.0', 'testStateNm': '완료', 'page': 1, 'sord': '', 'workplaceNm': 'ESNB', 'testNmEmpNo': '07005836', 'multiselectYn': 'N', 'testFullTitle': '07005836_20240531_JF1_DV2nd_no.2_BE27B13035_SOC0-100_0.25CP_25oC', 'jobCheckSkipYn': 'N', 'testChannelCnt': 1, 'iamTeamLeaderYn': 'N', 'iamTosOperLeaderYn': 'N', 'equipNm': 'APR-200-B232(HQ_ESS)', 'outterIp': '10.61.10.41', 'records': 0, 'chdchEquipI

In [38]:
import sys
# sys.path에 상위 폴더(..)를 추가합니다.
sys.path.append("..")
from src.voltaflow.connectors.minio import MinioConnector

def scan_minio(minio_connector):
    rows_to_insert = []
    res = minio_connector.client.list_objects_v2(Bucket="tos-stepend")

    for row in res['Contents']:
        if True: #<-TOS에서 데이터 다운로드 시작을 시작한 시간 뒤에 last modeified가 있으면서, required_downlod 컬럼이 True일 때! logic 추가 필요. 우선은 그냥 모든 파일을 가져오도록 합니다.
            rows_to_insert.append(row['Key'])
    return rows_to_insert

minio_connector = MinioConnector()
scan_minio(minio_connector)

Minio connection established successfully


['20250501_JF2S_CCV_T55_CPC_1_1_12CP_SOC0~100_UDKTF10115/EXP_250507_00840/07004515_20250501_JF2S_CCV_T55_CPC_1_1_12CP_SOC0~100_UDKTF10115.parquet',
 'CG02EC0281/EXP_230817_00443/07004515_20230817_JH4_MP_4M7_FR_No19_25_0.2CP_SOC50_CG02EC0281.parquet',
 'CG02EC0281/EXP_240703_00018/07004515_20230817_JH4_MP_4M7_FR_No19_25_0.2CP_SOC50_CG02EC0281.parquet',
 'CG02EC0281/EXP_240903_00075/07004195_20240903_JH4_MP_4M7_FR_No19_25_0.2CP_SOC50_CG02EC0281.parquet',
 'CG02EC0281/EXP_240930_00244/07004195_240930_JH4_FR_No19_CG02EC0281_Last RPT.parquet',
 'DA05B19192/EXP_240507_00736/07004515_20240507_JF1_MP_SII_JIS_45_0.5CCCV_0.5CC_DA05B19192.parquet',
 'DA05B19192/EXP_240711_00516/07004515_20240507_JF1_MP_SII_JIS_45_0.5CCCV_0.5CC_DA05B19192.parquet',
 'DA05B19192/EXP_240806_00487/07004515_20240806_JF1_MP_SII_JIS_45_0.5CCCV_0.5CC_DA05B19192.parquet',
 'DA05B19192/EXP_250213_00882/07004515_20250213_JF1_MP_SII_JIS_45_0.5CCCV_0.5CC_DA05B19192_re.parquet',
 'DA05B19192/EXP_250214_00027/07004515_20250214_

In [1]:
import requests
import json
import pandas as pd
import logging
import time
from datetime import  datetime
import os

In [29]:
df = pd.read_parquet(r"C:\Users\jhchoei\Desktop\Workspace\VoltaFlow\data\10-24-2025-15-00-51_files_list\EXP_250220_00022\07004515_20250219_JF2S_CV_ACC_TEST27_55_09045CP_1CP_UDKTF10021.parquet")

In [30]:
df

,StepType,Code,Fault,TotalCycle,StepNo,Temp,Current,Voltage,Capacity,Power,...,ChargeCVCap,WorkingTime,group_id,TotalTime_diff,TotalTime,StepTime_sec,StepTime,TestMode,IsRptCapa,IsRptDcir
0,Rest,Time Complete,0,1,2,58.5,0.0000,3.2755,0.000,0.000,...,0,14406,1,60.00,0:04:00:00,14400.00,04:00:00,RPT,False,False
1,Discharge,Voltage Complete,0,2,5,59.2,-38.9486,2.5000,44.697,-97.371,...,0,18539,2,52.57,0:05:08:52,4132.57,01:08:52,RPT,False,False
2,Rest,Time Complete,0,2,6,58.9,0.0000,2.7017,0.000,0.000,...,0,20339,3,60.00,0:05:38:52,1800.00,00:30:00,RPT,False,False
3,Charge,Current Complete,0,3,9,59.1,7.7920,3.6500,159.957,28.440,...,222,35154,4,54.22,0:09:45:46,14814.22,04:06:54,RPT,True,False
4,Rest,Time Complete,0,3,10,58.9,0.0000,3.5244,0.000,0.000,...,0,36354,5,60.00,0:10:05:46,1200.00,00:20:00,RPT,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487,Rest,Time Complete,0,99,27,59.0,0.0000,3.5279,0.000,0.000,...,0,1568941,393,60.00,18:03:47:03,1200.00,00:20:00,In-situ,False,False
488,Discharge,Voltage Complete,0,99,28,62.4,-199.4418,2.4999,155.290,-498.584,...,0,1572494,394,12.64,18:04:46:15,3552.64,00:59:12,In-situ,False,False
489,Rest,Time Complete,0,99,29,58.7,0.0000,2.8151,0.000,0.000,...,0,1576095,395,60.00,18:05:46:15,3600.00,01:00:00,In-situ,False,False
490,Charge,Voltage Complete,0,100,25,59.6,133.5141,3.3610,35.648,448.740,...,0,1577031,396,36.28,18:06:01:52,936.28,00:15:36,In-situ,False,False
